In [1]:
import os
import json
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("Groq model initialized successfully.")

Groq model initialized successfully.


In [2]:
support_chat = """
Customer: My online payment was deducted, but my order was not placed.
Support Agent: Please share your order ID.
Customer: I do not have an order ID because the order was never created.
Support Agent: Please wait while I check.
Customer: I have already waited 48 hours, and the money has not been refunded.
Support Agent: I will look into it.
Customer: I need this resolved urgently.
"""

print(support_chat)


Customer: My online payment was deducted, but my order was not placed.
Support Agent: Please share your order ID.
Customer: I do not have an order ID because the order was never created.
Support Agent: Please wait while I check.
Customer: I have already waited 48 hours, and the money has not been refunded.
Support Agent: I will look into it.
Customer: I need this resolved urgently.



In [3]:
class SupportSummary(BaseModel):
    customer_issue: str = Field(description="Main issue reported by the customer")
    actions_taken: list[str] = Field(description="Actions taken by support")
    customer_impact: str = Field(description="Impact or inconvenience to the customer")
    unresolved_items: list[str] = Field(description="Issues that remain unresolved")
    urgency: str = Field(description="Urgency: Low, Medium, or High")


summary_parser = PydanticOutputParser(pydantic_object=SupportSummary)

print("Support summary schema created.")

Support summary schema created.


In [4]:
summary_prompt = PromptTemplate(
    template="""
Analyze the support conversation and extract an accurate structured summary.

Do not invent actions, facts, or resolutions.
If an action is not clearly completed, do not claim it was completed.
Use High, Medium, or Low for urgency.

Support conversation:
{support_chat}

{format_instructions}
""",
    input_variables=["support_chat"],
    partial_variables={
        "format_instructions": summary_parser.get_format_instructions()
    }
)

summary_chain = summary_prompt | llm | summary_parser

summary = summary_chain.invoke({"support_chat": support_chat})

print(summary.model_dump_json(indent=2))

{
  "customer_issue": "Payment was deducted online but the order was never created, and the customer has not received a refund.",
  "actions_taken": [
    "Requested the order ID from the customer",
    "Informed the customer that they would check the transaction",
    "Stated they would look into the missing order and refund"
  ],
  "customer_impact": "The customer lost money without receiving the purchased product, has been waiting 48 hours for a refund, and is experiencing financial inconvenience.",
  "unresolved_items": [
    "Order was not created",
    "Refund has not been issued",
    "No resolution or confirmation provided"
  ],
  "urgency": "High"
}


In [5]:
class EscalationGapCheck(BaseModel):
    escalation_needed: bool = Field(
        description="Whether the case should be escalated"
    )
    missing_actions: list[str] = Field(
        description="Important actions not completed or not confirmed"
    )
    reasons: list[str] = Field(
        description="Evidence-based reasons for escalation"
    )


gap_parser = PydanticOutputParser(
    pydantic_object=EscalationGapCheck
)

gap_prompt = PromptTemplate(
    template="""
Review the structured support summary.

Escalate when an issue remains unresolved and requires further investigation.
Distinguish between actions that were completed and actions merely promised.
Do not assume that a refund was processed or that an investigation was completed.

Structured summary:
{summary_json}

{format_instructions}
""",
    input_variables=["summary_json"],
    partial_variables={
        "format_instructions": gap_parser.get_format_instructions()
    }
)

gap_chain = gap_prompt | llm | gap_parser

gap_check = gap_chain.invoke({
    "summary_json": summary.model_dump_json()
})

print(gap_check.model_dump_json(indent=2))

{
  "escalation_needed": true,
  "missing_actions": [
    "Create the missing order in the system",
    "Process and confirm the refund to the customer",
    "Verify the transaction details with the payment processor",
    "Provide the customer with a confirmation of the refund and order status",
    "Document the investigation outcome and communicate it to the customer"
  ],
  "reasons": [
    "The customer's payment was deducted but no order was created, resulting in a loss of money without product delivery.",
    "48 hours have passed without a refund or any concrete resolution, indicating the issue remains unresolved.",
    "All actions taken so far are only promises or requests; no concrete remediation (order creation or refund) has been completed.",
    "The issue is marked as high urgency due to financial inconvenience to the customer.",
    "Further investigation is required to determine why the order was not created and why the refund has not been issued."
  ]
}


In [6]:
report_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a customer-support escalation specialist. "
        "Write a concise, professional escalation report using only "
        "the supplied structured summary and gap check. "
        "Do not invent transaction IDs, refund status, or completed actions."
    ),
    (
        "human",
        """
Create an escalation report with these sections:
1. Customer Issue
2. Actions Already Taken
3. Outstanding Gaps
4. Reason for Escalation
5. Recommended Next Steps

Structured Summary:
{summary_json}

Escalation Gap Check:
{gap_json}
"""
    )
])

report_chain = report_prompt | llm | StrOutputParser()

escalation_report = report_chain.invoke({
    "summary_json": summary.model_dump_json(indent=2),
    "gap_json": gap_check.model_dump_json(indent=2)
})

print(escalation_report)

**Escalation Report**

---

### 1. Customer Issue  
The customer’s payment was deducted online, but no order was created and no refund has been received. The customer has been waiting 48 hours for a resolution, resulting in financial inconvenience.

### 2. Actions Already Taken  
- Requested the order ID from the customer.  
- Informed the customer that the transaction would be reviewed.  
- Communicated that the missing order and refund would be investigated.

### 3. Outstanding Gaps  
- Missing order record in the system.  
- Refund has not been processed or confirmed.  
- Transaction details have not been verified with the payment processor.  
- No confirmation of order status or refund has been provided to the customer.  
- Investigation outcome has not been documented or communicated.

### 4. Reason for Escalation  
- Payment was taken without product delivery, causing a direct monetary loss.  
- High‑urgency timeframe (48 hours) has elapsed with no concrete remediation.  
- All p

In [7]:
assert isinstance(summary, SupportSummary)
assert summary.customer_issue.strip()
assert isinstance(summary.actions_taken, list)
assert isinstance(summary.unresolved_items, list)

assert isinstance(gap_check, EscalationGapCheck)
assert gap_check.escalation_needed is True

assert isinstance(escalation_report, str)
assert escalation_report.strip()

print("Validation passed.")
print("Structured summary generated.")
print("Escalation gap check completed.")
print("Escalation report generated.")

Validation passed.
Structured summary generated.
Escalation gap check completed.
Escalation report generated.


In [8]:
print("Exercise 5 completed successfully.")
print("The support conversation was summarized, checked for escalation gaps,")
print("and converted into a professional escalation report.")

Exercise 5 completed successfully.
The support conversation was summarized, checked for escalation gaps,
and converted into a professional escalation report.
